# M6A3 - Sistemas de Monitoramento de Experimentos

Na prática de hoje vamos monitorar um experimento utilizando o [TensorBoard](https://www.tensorflow.org/tensorboard?hl=pt-br).

Esse notebook está estruturado da seguinte forma.

- Introdução
- Rodar experimento simples
- Acompanhar logs
- Próximos passos
- Atividade Complementares

## Introdução

Instalação para os que ainda não possuem a biblioteca instalada.

In [4]:
!pip install torch torchvision tensorboard

Importar as bibliotecas

In [5]:
import torch
import torchvision
from torch.utils.tensorboard import SummaryWriter


## Rodar Experimentos Simples

Para isso iremos reproduzir o experimento da aula M4A2 sobre GANs.

In [ ]:
####################
## Carregar Dados ##
####################

batch_size = 100

# MNIST Dataset
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=(0.5), std=(0.5))])

train_dataset = torchvision.datasets.MNIST(root='./mnist_data/', train=True, transform=transform, download=True)
# Data Loader (Input Pipeline)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)


########################
## Criando os modelos ##
########################


# Gerador.
class Generator(torch.nn.Module):
    def __init__(self, g_input_dim, g_output_dim):
        super(Generator, self).__init__()       
        self.fc1 = torch.nn.Linear(g_input_dim, 256)
        self.fc2 = torch.nn.Linear(self.fc1.out_features, self.fc1.out_features*2)
        self.fc3 = torch.nn.Linear(self.fc2.out_features, self.fc2.out_features*2)
        self.fc4 = torch.nn.Linear(self.fc3.out_features, g_output_dim)
    
    # método forward. 
    def forward(self, x): 
        x = torch.nn.functional.leaky_relu(self.fc1(x), 0.2)
        x = torch.nn.functional.leaky_relu(self.fc2(x), 0.2)
        x = torch.nn.functional.leaky_relu(self.fc3(x), 0.2)
        return torch.tanh(self.fc4(x))

# Discrimador.
class Discriminator(torch.nn.Module):
    def __init__(self, d_input_dim):
        super(Discriminator, self).__init__()
        self.fc1 = torch.nn.Linear(d_input_dim, 1024)
        self.fc2 = torch.nn.Linear(self.fc1.out_features, self.fc1.out_features//2)
        self.fc3 = torch.nn.Linear(self.fc2.out_features, self.fc2.out_features//2)
        self.fc4 = torch.nn.Linear(self.fc3.out_features, 1)
    
    # método forward. 
    def forward(self, x):
        x = torch.nn.functional.leaky_relu(self.fc1(x), 0.2)
        x = torch.nn.functional.dropout(x, 0.3)
        x = torch.nn.functional.leaky_relu(self.fc2(x), 0.2)
        x = torch.nn.functional.dropout(x, 0.3)
        x = torch.nn.functional.leaky_relu(self.fc3(x), 0.2)
        x = torch.nn.functional.dropout(x, 0.3)
        return torch.sigmoid(self.fc4(x))
    

#################
## Treinamento ##
#################

# Instanciar o logger do Tensorboard.
# Pode alterar o path.
writer =  SummaryWriter("logs/gan_2")

# Instanciar as redes.
z_dim = 100
mnist_dim = train_dataset.train_data.size(1) * train_dataset.train_data.size(2)

device = "cuda" if torch.cuda.is_available() else "cpu"
G = Generator(g_input_dim = z_dim, g_output_dim = mnist_dim).to(device)
D = Discriminator(mnist_dim).to(device)

# Função de perda.
criterion = torch.nn.BCELoss() 

# Otimizador.
lr = 0.0002 
G_optimizer = torch.optim.Adam(G.parameters(), lr = lr)
D_optimizer = torch.optim.Adam(D.parameters(), lr = lr)

def D_train(x):
    #=======================Treino do discriminador=======================#
    D.zero_grad()

    # Treina discriminador em dados reais.
    x_real, y_real = x.view(-1, mnist_dim), torch.ones(batch_size, 1)
    x_real, y_real = torch.autograd.Variable(x_real.to(device)), torch.autograd.Variable(y_real.to(device))

    D_output = D(x_real)
    D_real_loss = criterion(D_output, y_real)
    D_real_score = D_output

    # Treina discriminador em dados falsos.
    z = torch.autograd.Variable(torch.randn(batch_size, z_dim).to(device))
    x_fake, y_fake = G(z), torch.autograd.Variable(torch.zeros(batch_size, 1).to(device))

    D_output = D(x_fake)
    D_fake_loss = criterion(D_output, y_fake)
    D_fake_score = D_output

    # Backpropagation e otimização dos parâmetros do discriminador.
    D_loss = D_real_loss + D_fake_loss
    D_loss.backward()
    D_optimizer.step()
        
    return  D_loss.data.item()

def G_train(x):
    #=======================Treino do gerador=======================#
    G.zero_grad()

    z = torch.autograd.Variable(torch.randn(batch_size, z_dim).to(device))
    y = torch.autograd.Variable(torch.ones(batch_size, 1).to(device))

    G_output = G(z)
    D_output = D(G_output)
    G_loss = criterion(D_output, y)

    # Backpropagation e otimização dos parâmetros do gerador.
    G_loss.backward()
    G_optimizer.step()
        
    return G_loss.data.item()

# Laço de treino.
n_epoch = 200
for epoch in range(1, n_epoch+1):           
    D_losses, G_losses = [], []
    for batch_idx, (x, _) in enumerate(train_loader):
        D_losses.append(D_train(x))
        G_losses.append(G_train(x))
        writer.add_scalar("Loss/Treino_Discriminator_step", D_losses[-1], batch_idx)
        writer.add_scalar("Loss/Treino_Generator_step", G_losses[-1], batch_idx)

    print('[%d/%d]: loss_d: %.3f, loss_g: %.3f' % (
            (epoch), n_epoch, torch.mean(torch.FloatTensor(D_losses)), torch.mean(torch.FloatTensor(G_losses))))
    writer.add_scalar("Loss/Treino_Discriminator_epoch", torch.mean(torch.FloatTensor(D_losses)), epoch)
    writer.add_scalar("Loss/Treino_Generator_epoch", torch.mean(torch.FloatTensor(G_losses)), epoch)
    
    # Gerando imagens para os logs.
    with torch.no_grad():
        test_z = torch.autograd.Variable(torch.randn(batch_size, z_dim).to(device))
        generated = G(test_z)

    generated = generated.view(generated.size(0), 1, 28, 28)
    grid = torchvision.utils.make_grid(generated.cpu(), 20)
    writer.add_image("Imagens Geradas", grid, epoch)

writer.flush()
writer.close()

[1/10]: loss_d: 0.834, loss_g: 3.159
[2/10]: loss_d: 1.009, loss_g: 2.386
[3/10]: loss_d: 0.819, loss_g: 1.900
[4/10]: loss_d: 0.446, loss_g: 3.547
[5/10]: loss_d: 0.485, loss_g: 3.107
[6/10]: loss_d: 0.573, loss_g: 2.701
[7/10]: loss_d: 0.529, loss_g: 2.979
[8/10]: loss_d: 0.557, loss_g: 2.646
[9/10]: loss_d: 0.567, loss_g: 2.683
[10/10]: loss_d: 0.659, loss_g: 2.400


## Acompanhar Logs

Para isso rodamos o seguinte comando.

In [7]:
!tensorboard --logdir logs

TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

I0325 12:15:48.356192 126089158452800 plugin.py:429] Monitor runs begin
Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.20.0 at http://localhost:6006/ (Press CTRL+C to quit)
^C


## Próximos Passos e Referências

E essa é a última prática da nossa trilha de Visão Computacional, espero que tenho aproveitado.

Uma lista não exaustiva de referências segue:

- https://www.tensorflow.org/tensorboard?hl=pt-br
- https://docs.pytorch.org/docs/stable/tensorboard.html
- https://github.com/lyeoni/pytorch-mnist-GAN
- https://pytorch.org/
- https://docs.pytorch.org/vision/main/models.html
- https://opencv.org/
- https://learnopencv.com/blogs/
- https://pyimagesearch.com/

## Atividades Complementares (Opcional)

- [ ] Explore outras funções de logs possíveis no tensorboard e veja os logs.
- [ ] Existem outras soluções para controle de experimentos?